# 0 - Null Analysis

Analisis interactivo de valores nulos en los datasets finales del ETL.
Carga los parquets de `data/data_cleaned/` y usa `NullValidator` para
explorar la distribucion de nulos por columna y verificar umbrales esperados.

**Prerequisito:** ejecutar el ETL antes de abrir este notebook:
```bash
python -m src.pipeline
```

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.utils.nulls import NullValidator

DATA_CLEANED = ROOT / 'data' / 'data_cleaned'

df_barrios   = pd.read_parquet(DATA_CLEANED / 'df_barrios_eda.parquet')
df_distritos = pd.read_parquet(DATA_CLEANED / 'df_distritos_eda.parquet')

print(f'df_barrios:   {df_barrios.shape}')
print(f'df_distritos: {df_distritos.shape}')

df_barrios:   (3102, 29)
df_distritos: (1030, 28)


## Barrios

In [2]:
v = NullValidator()
report_barrios = v.report(df_barrios, 'barrios')


  NULL REPORT — barrios (3102 filas)  
                  columna  n_nulos   pct  esperado_max_%   ok
        neighborhood_code        0  0.00             NaN None
             neighborhood        0  0.00             NaN None
                     year        0  0.00             NaN None
                  quarter        0  0.00             NaN None
            num_contracts        0  0.00             0.0 True
                 avg_rent        0  0.00             0.0 True
              avg_rent_m2        0  0.00             0.0 True
              avg_surface        0  0.00             0.0 True
                     date        0  0.00             NaN None
              ipc_index_q        0  0.00            10.0 True
            euribor_12m_q        0  0.00             0.0 True
                ist_index      726 23.40            30.0 True
      contract_growth_yoy      264  8.51            10.0 True
          rent_growth_yoy      264  8.51            10.0 True
       rent_m2_growth_yoy     

## Distritos

In [3]:
report_distritos = v.report(df_distritos, 'distritos')


  NULL REPORT — distritos (1030 filas)  
                  columna  n_nulos   pct  esperado_max_%   ok
            district_code        0  0.00             NaN None
                 district        0  0.00             NaN None
                     year        0  0.00             NaN None
                  quarter        0  0.00             NaN None
            num_contracts        0  0.00             0.0 True
                 avg_rent        0  0.00             0.0 True
              avg_rent_m2        0  0.00             0.0 True
              avg_surface        0  0.00             0.0 True
                     date        0  0.00             NaN None
              ipc_index_q       80  7.77            10.0 True
            euribor_12m_q        0  0.00             0.0 True
      contract_growth_yoy       40  3.88            10.0 True
          rent_growth_yoy       40  3.88            10.0 True
       rent_m2_growth_yoy       40  3.88            10.0 True
      contract_growth_qoq   

## Validacion contra umbrales esperados

In [4]:
v.assert_within_thresholds(df_barrios,   'barrios')
v.assert_within_thresholds(df_distritos, 'distritos')

[barrios] Validación de nulos OK.
[distritos] Validación de nulos OK.


## Exploracion libre

Inspecciona columnas con nulos de forma detallada.

In [5]:
cols_con_nulos = report_barrios[report_barrios['n_nulos'] > 0]['columna'].tolist()
print('Columnas con nulos en barrios:', cols_con_nulos)

# Distribucion de nulos en ist_index por anio (estructural: IST solo 2015-2024)
if 'ist_index' in df_barrios.columns:
    print('\nNulos en ist_index por anio:')
    print(
        df_barrios.groupby('year')['ist_index']
        .apply(lambda x: x.isna().sum())
        .rename('n_nulos')
        .to_string()
    )

Columnas con nulos en barrios: ['ist_index', 'contract_growth_yoy', 'rent_growth_yoy', 'rent_m2_growth_yoy', 'contract_growth_qoq', 'rent_m2_lag1', 'rent_m2_lag4', 'rent_real_growth_yoy', 'rent_m2_real_growth_yoy']

Nulos en ist_index por anio:
year
2014    264
2015      0
2016      0
2017      0
2018      0
2019      0
2020      0
2021      0
2022      0
2023      0
2024    264
2025    198


In [6]:
v.validate_no_duplicates(df_barrios,   ['neighborhood_code', 'year', 'quarter'], 'barrios')
v.validate_no_duplicates(df_distritos, ['district_code',     'year', 'quarter'], 'distritos')

[barrios] Sin duplicados en ['neighborhood_code', 'year', 'quarter']. OK
[distritos] Sin duplicados en ['district_code', 'year', 'quarter']. OK
